*0.1 Python for GenAI*

# Typing

**The situation.** A refactor renames the `content` field of chat messages to `text`. The developer updates forty places. The forty-first is an export job that runs once a month, and it still reads `message["content"]`. All tests pass. The export fails on the first of next month, in production.

**The fix: write the shape next to the code, and let a tool check it.** A *type hint* is a label: `name: str` says "name holds text". Python ignores the labels when it runs. A checker tool — pyright — reads them and reports every place where the label and the use disagree, before the code runs.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The shape of a chat message.** `TypedDict` describes a dictionary's keys and value types. `Literal` restricts a value to a fixed set.

In [2]:
from typing import Literal, TypedDict


class Message(TypedDict):
    role: Literal["system", "user", "assistant"]  # only these three values are allowed
    content: str


def build_messages(system: str, user: str) -> list[Message]:
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


messages = build_messages("You are a support assistant.", "How do I reset my password?")
print(messages)
assert messages[1]["role"] == "user"

[{'role': 'system', 'content': 'You are a support assistant.'}, {'role': 'user', 'content': 'How do I reset my password?'}]


**Now the export job, with the bug.** This file uses the old field name and also has a second mistake. Python will happily run it — until the line executes. Run pyright on it instead.

In [3]:
import json
import subprocess
import tempfile
from pathlib import Path

export_job = """
from typing import Literal, TypedDict

class Message(TypedDict):
    role: Literal["system", "user", "assistant"]
    text: str                                   # renamed from "content"

def export(messages: list[Message]) -> list[str]:
    lines = []
    for message in messages:
        lines.append(message["content"])        # bug 1: the old field name
    return lines

count: int = "3"                                # bug 2: text stored in an int
"""
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "export_job.py"
    path.write_text(export_job)
    report = subprocess.run(
        ["uv", "run", "pyright", "--outputjson", str(path)], capture_output=True, text=True
    )
problems = json.loads(report.stdout)["generalDiagnostics"]
for problem in problems:
    print(
        f"line {problem['range']['start']['line'] + 1}: {problem['message'].splitlines()[0][:90]}"
    )
assert len(problems) == 2

line 11: Could not access item in TypedDict
line 14: Type "Literal['3']" is not assignable to declared type "int"


**Reading the output.** pyright found both bugs and named the lines — without running the code. In a real project this runs on every pull request and fails the build, so the forty-first call site is caught before it is merged.

**The rule to remember.** Type the shapes of your data (messages, records, results) and run the checker in CI. It is the cheapest test you will ever write.

| Use it when | Don't when | Instead use |
|---|---|---|
| any code with more than one author or more than a week of life | a throwaway script | — |

**Watch out**
- Writing `Any` to make the checker quiet removes the protection exactly where you needed it.
- Data from a provider's API is untyped. Turn it into a Pydantic model at the edge; everything inside is then typed.
- A checker that only prints warnings is decoration. Make it fail the build.